# Issue 2:
Clean nested columns in `movies_metadata.csv`


In [1]:
from pathlib import Path
import pandas as pd
import ast

raw_data = Path("../../data/raw/movies_metadata.csv")
clean_data = Path("../../data/processed/nested_metadata.csv")

In [2]:
df = pd.read_csv(raw_data)
print(df.shape)
print()
df.head()

(45466, 24)



/tmp/ipykernel_32733/788480134.py:1: DtypeWarning: Columns (10: popularity) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(raw_data)


,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0


## belongs_to_collection

In [3]:
test = df["belongs_to_collection"].iloc[0]
print(test)
print(type(test))

{'id': 10194, 'name': 'Toy Story Collection', 'poster_path': '/7G9915LfUQ2lVfwMEEhDsn3kT4B.jpg', 'backdrop_path': '/9FBwqcd9IRruEDUrTdcaafOMKUq.jpg'}
<class 'str'>


Convert to dictionary using `ast.literal_eval()`, extract the name of collection, then apply it in place. If the cell containted invalid data (ex. not a string), replace it with NA. Some cells had numerical values and nothing could be extracted; therefore, setting to NA makes the most sense.

In [4]:
for i in df["belongs_to_collection"].index:
    string = df['belongs_to_collection'].iloc[i]
    if pd.notna(string):
        conversion = ast.literal_eval(string)
        if isinstance(conversion, dict):
            df.at[i, 'belongs_to_collection'] = conversion["name"]
        else:
            df.at[i, 'belongs_to_collection'] = pd.NA


df["belongs_to_collection"].head()


0              Toy Story Collection
1                               NaN
2         Grumpy Old Men Collection
3                               NaN
4    Father of the Bride Collection
Name: belongs_to_collection, dtype: str

# production_companies

In [5]:
test = df["production_companies"].iloc[1]
print(test)
print(type(test))

[{'name': 'TriStar Pictures', 'id': 559}, {'name': 'Teitler Film', 'id': 2550}, {'name': 'Interscope Communications', 'id': 10201}]
<class 'str'>


A similar approach can be used as belongs_to_collection. However, it must be adapted since we now have a list of dictionaries, rather than just a dictionary.

In [6]:
df["production_companies"] = df["production_companies"].astype(object)

for i in df["production_companies"].index:
    string = df['production_companies'].iloc[i]
    if pd.notna(string):
        conversion = ast.literal_eval(string)
        if isinstance(conversion, list):
            lst = []
            for element in conversion:
                if element['name'] not in lst:
                    lst.append(element['name'])
            df.at[i, "production_companies"] = "|".join(lst)
        else:
            df.at[i, 'production_companies'] = pd.NA
            
df["production_companies"].head()


0                              Pixar Animation Studios
1    TriStar Pictures|Teitler Film|Interscope Commu...
2                          Warner Bros.|Lancaster Gate
3               Twentieth Century Fox Film Corporation
4            Sandollar Productions|Touchstone Pictures
Name: production_companies, dtype: object

# release_date

In [7]:
df['release_date'].head()

0    1995-10-30
1    1995-12-15
2    1995-12-22
3    1995-12-22
4    1995-02-10
Name: release_date, dtype: str

In [8]:
converted_dates = pd.to_datetime(df["release_date"], errors="coerce")
print(df[converted_dates.isna()][['original_title', 'release_date']])


                                       original_title release_date
711              War Stories Our Mother Never Told Us          NaN
734                             Vermont Is for Lovers          NaN
3460                       Jails, Hospitals & Hip-Hop          NaN
3628                                   Boricua's Bond          NaN
5879                              Divine Intervention          NaN
...                                               ...          ...
45148               Konstruktor krasnogo tsveta -1993          NaN
45203  All Superheroes Must Die 2: The Last Superhero          NaN
45338                  The Land Where the Blues Began          NaN
45410                                           Aprel          NaN
45461                                         رگ خواب          NaN

[90 rows x 2 columns]
